<a href="https://colab.research.google.com/github/jvonrad/Lost-in-Mistranslation/blob/main/data_analysis/Lost_In_Multilinguality.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Clear the hidden cache folder
!rm -rf /root/.cache/huggingface/hub

# 2. Check your disk space to confirm it worked
!df -h / | grep "/"

## First Time Setup

In [ ]:
# necessary installs
!pip install -q -U bitsandbytes>=0.46.1 accelerate transformers huggingface_hub

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from google.colab import drive
# import os

In [ ]:
# !rm -rf /content/drive

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# # import torch
# # import matplotlib.pyplot as plt
# # from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
# from tqdm import tqdm

# # 4bit quantize model
# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_use_double_quant=True,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_compute_dtype=torch.bfloat16
# )

# max_memory = {0: "8GiB", "cpu": "20GiB"}

# # --- Model Identifiers ---
# base_id = "allenai/OLMo-2-1124-7B"
# # ft_id = "jonny-vr/olmo2-7b-ted-fullft"
# # ft_lora_id_old = "jonny-vr/olmo2-7b-ted-lora"
# # ft_lora_id = "GBBurgess/olmo2-lora-ft"

# # Load Tokenizer (usually shared)
# tokenizer = AutoTokenizer.from_pretrained(base_id)

# # Load Models
# model_base = AutoModelForCausalLM.from_pretrained(
#     base_id, # base_id,
#     quantization_config=bnb_config,
#     device_map="auto", # Moves shards to GPU immediately to save System RAM
#     max_memory=max_memory,
#     low_cpu_mem_usage=True,
#     trust_remote_code=True,
#     offload_folder='offload',
#     offload_state_dict=True
#     )
# # model_ft = AutoModelForCausalLM.from_pretrained(ft_id, torch_dtype=torch.bfloat16, device_map="auto")
# # model_ft_lora = AutoModelForCausalLM.from_pretrained(ft_lora_id, torch_dtype=torch.bfloat16, device_map="auto")

In [ ]:
# define paths
save_path = "/content/drive/MyDrive/SNLP_Project/models/olmo2_base"
ft_save_path = "/content/drive/MyDrive/SNLP_Project/models/olmo2_ft_full"
ft_lora_save_path = "/content/drive/MyDrive/SNLP_Project/models/olmo2_ft_lora"

In [ ]:
# # save models to drive (once)
# model_base.save_pretrained(save_path)
# tokenizer.save_pretrained(save_path)

# # model_base.save_pretrained(ft_save_path)
# # tokenizer.save_pretrained(ft_save_path)

# # model_ft_lora.save_pretrained(ft_lora_save_path)
# # tokenizer.save_pretrained(ft_lora_save_path)

In [ ]:
# reset cache
del model_base
del tokenizer
torch.cuda.empty_cache()

!rm -rf /root/.cache/huggingface/hub

## Copy Files to Drive, Load from drive and quantize and save quantized to Drive (one time per model)

In [ ]:
from huggingface_hub import snapshot_download
from google.colab import drive
import os

drive.mount('/content/drive', force_remount=True)

# Define where the RAW (30GB) files will live on your Drive
raw_base_path = "/content/drive/MyDrive/SNLP_Project/models/olmo2_lora_ft_raw"

print("⏳ Downloading raw files directly to Drive (This is the last time you'll wait 20 mins)...")

snapshot_download(
    repo_id="GBBurgess/olmo2-lora-ft",
    local_dir=raw_base_path,
    local_dir_use_symlinks=False,
    ignore_patterns=["*.msgpack", "*.h5"] # Only get the safetensors/config
)

print(f"✅ DONE! Raw files are safe at {raw_base_path}. You never have to download this again.")

In [ ]:
from google.colab import drive

# 1. Mount Drive
drive.mount('/content/drive', force_remount=True)

# Create a local folder on the Colab SSD
!mkdir -p /content/olmo2_lora_ft_raw

# Copy from Drive to Local (FAST)
print("⏳ Copying weights to local SSD...")
!cp -r /content/drive/MyDrive/SNLP_Project/models/olmo2_lora_ft_raw/* /content/olmo2_lora_ft_raw/
print("✅ Done! Files are now local.")

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# 1. Point to the LOCAL folder, not the Drive folder
local_path = "/content/olmo2_lora_ft_raw"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

# Use the GPU! Since it's local, it won't time out.
model = AutoModelForCausalLM.from_pretrained(
    local_path,
    quantization_config=bnb_config,
    device_map="auto",
    low_cpu_mem_usage=True,
    trust_remote_code=True
)
tokenizer = AutoTokenizer.from_pretrained(local_path)
print("🚀 Model loaded in minutes, not hours!")

In [ ]:
# Save the small, fast version to Drive
quant_save_path = "/content/drive/MyDrive/SNLP_Project/models/olmo2_lora_ft_4bit"

print(f"💾 Saving 4-bit version to Drive...")
model.save_pretrained(quant_save_path)
tokenizer.save_pretrained(quant_save_path)

print("✅ DONE! In the future, you can load from 'olmo2_base_4bit' in seconds.")

In [ ]:
!rm -rf /content/olmo2_lora_ft_raw

# Reload Model

I am losing my mind

In [ ]:
!pip install -q -U bitsandbytes>=0.46.1 accelerate transformers huggingface_hub

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# from huggingface_hub import notebook_login
# notebook_login()

In [ ]:
import torch
import gc
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch.nn.functional as F
import os
import json
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import re
import random
from collections import defaultdict
import unicodedata

In [ ]:
# --- UTILS ---
def clean_str(s):
    if not s: return ""
    return unicodedata.normalize('NFC', str(s)).lower().strip()

def overlapping_ratio(list1, list2):
    set1, set2 = set(list1), set(list2)
    intersection = set1.intersection(set2)
    union = set1.union(set2)
    return len(intersection) / len(union) if union else 0

def find_valid_specimens(specimens_list, target_type='success'):
    valid = []
    for i, s in enumerate(specimens_list):
        if s['type'] != target_type: continue

        id_nat = tokenizer.encode(" " + str(s["target_nat"]).strip(), add_special_tokens=False)[-1]
        id_eng = tokenizer.encode(" " + str(s["target_en"]).strip(), add_special_tokens=False)[-1]

        if id_nat != id_eng:
            valid.append((i, s))

        if len(valid) >= 5: break
    return valid

def get_layer_consistency(en_text, target_text, model, tokenizer):
    """
    Measures the cosine similarity between the English and Native hidden states
    at every layer of the model.
    """
    # 1. Tokenize both prompts
    inputs_en = tokenizer(en_text, return_tensors="pt").to("cuda")
    inputs_target = tokenizer(target_text, return_tensors="pt").to("cuda")

    with torch.no_grad():
        # 2. Run a forward pass and request all hidden states
        out_en = model(inputs_en.input_ids, output_hidden_states=True)
        out_target = model(inputs_target.input_ids, output_hidden_states=True)

    similarities = []

    # 3. Iterate through every layer (usually 28 or 32 layers)
    # hidden_states is a tuple of (embeddings, layer_1, layer_2, ..., layer_N)
    for h_en, h_target in zip(out_en.hidden_states, out_target.hidden_states):

        # 4. Extract the 'Factual' vector
        # We use the hidden state of the LAST token (-1) as it contains
        # the accumulated context of the whole prompt.
        v_en = h_en[0, -1, :]
        v_target = h_target[0, -1, :]

        # 5. Calculate Cosine Similarity
        # Result is a value between -1 (opposite) and 1 (identical)
        sim = F.cosine_similarity(v_en.unsqueeze(0), v_target.unsqueeze(0))
        similarities.append(sim.item())

    return similarities

# --- MASTER EVALUATION FUNCTION ---
def run_model_evaluation(model, tokenizer, model_label, n_shot=3, debug=True):
    print(f"\n🚀 STARTING EVALUATION: {model_label}")

    all_test_samples = []
    # --- CRITICAL FIX: Ensure subject_en is captured for the Curve ---
    target_langs = ['es'] if debug==True else TARGET_LANGS
    for lang in target_langs:
        lang_dir = os.path.join(DATA_ROOT, lang)
        if not os.path.exists(lang_dir): continue
        for f_name in sorted(os.listdir(lang_dir)):
            if not f_name.endswith(".json"): continue
            with open(os.path.join(lang_dir, f_name), 'r', encoding='utf-8') as f:
                data = json.load(f)
                template = data.get('prompt_templates', ["<subject> is"])[0]
                clean_template = re.sub(r'<mask.*?>', '', template).replace('.', '').strip()
                rel_en = f_name.replace('.json','').replace('_',' ')

                for s in data.get('samples', []):
                    all_test_samples.append({
                        "index": s.get("index"),
                        "language": lang,
                        "relation_en": rel_en,
                        "subject_en": s.get("subject_en"),
                        "input": clean_template.replace('<subject>', s['subject']),
                        "target": s.get('object') or s.get('target'),
                        "target_en": s.get('object_en') or s.get('target_en') # <--- ADD THIS LINE
                    })

    per_lang_results = defaultdict(lambda: {"correct_indices": [], "total": 0, "curves": []})

    specimens = []
    for lang in target_langs:
        lang_data = [s for s in all_test_samples if s['language'] == lang]
        if not lang_data: continue
        eval_subset = lang_data[:15] if debug else lang_data

        for ex in tqdm(eval_subset, desc=f"Probing {lang.upper()}"):
            # 1. FEW-SHOT GENERATION (For Figure 8)
            candidates = [c for c in lang_data if c['index'] != ex['index'] and c['relation_en'] == ex['relation_en']]
            demonstrations = random.sample(candidates, min(n_shot, len(candidates)))
            few_shot_prompt = "".join([f"{d['input']} {d['target']}\n" for d in demonstrations]) + ex["input"] + " "

            inputs = tokenizer(few_shot_prompt, return_tensors="pt").to("cuda")
            with torch.no_grad():
                output = model.generate(**inputs, max_new_tokens=10, do_sample=False, pad_token_id=tokenizer.eos_token_id)

            pred = clean_str(tokenizer.decode(output[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True).split('\n')[0])
            target = clean_str(ex["target"])

            if target in pred or (len(target) > 3 and pred.startswith(target[:4])):
                per_lang_results[lang]["correct_indices"].append(ex["index"])
            per_lang_results[lang]["total"] += 1

            if target in pred or (len(target) > 3 and pred.startswith(target[:4])):
                per_lang_results[lang]["correct_indices"].append(ex["index"])
                match = True  # Helper variable for the collector
            else:
                match = False

            per_lang_results[lang]["total"] += 1

            # 1. Capture SUCCESS specimens (for Figure 5a)
            if match and len([s for s in specimens if s['type'] == 'success' and s['lang'] == lang]) < 3:
                specimens.append({
                    "type": "success",
                    "lang": lang,
                    "relation": ex["relation_en"],
                    "prompt": few_shot_prompt,
                    "target_nat": ex["target"],
                    "target_en": ex["target_en"],
                    "subject": ex["subject_en"]
                })

            # 2. Capture FAILURE specimens (for Figure 5b)
            elif not match and len(pred.strip()) > 3 and len([s for s in specimens if s['type'] == 'failure' and s['lang'] == lang]) < 3:
                specimens.append({
                    "type": "failure",
                    "lang": lang,
                    "relation": ex["relation_en"],
                    "prompt": few_shot_prompt,
                    "target_nat": ex["target"],
                    "model_pred": pred,
                    "target_en": ex["target_en"],
                    "subject": ex["subject_en"]
                })

            # 2. INTERNAL CONSISTENCY PROBE (For Figure 5)
            # Compare: "The capital of France is" vs "La capitale de la France est"
            if ex["subject_en"]:
                en_p = f"The {ex['relation_en']} of {ex['subject_en']} is"
                nat_p = ex["input"]

                # Directly calling the function and appending
                curve = get_layer_consistency(en_p, nat_p, model, tokenizer)
                per_lang_results[lang]["curves"].append(curve)

    # 3. CONSOLIDATE
    final_stats = {}
    for lang in per_lang_results:
        final_stats[lang] = {
            "acc": len(per_lang_results[lang]["correct_indices"]) / max(1, per_lang_results[lang]["total"]),
            "clc": 0, # Placeholder for CLC loop
            "avg_curve": np.mean(per_lang_results[lang]["curves"], axis=0).tolist() if per_lang_results[lang]["curves"] else []
        }

    # Calculate CLC Overlap
    langs = sorted(list(final_stats.keys()))
    for lang in langs:
        others = [l for l in langs if l != lang]
        overlaps = [overlapping_ratio(per_lang_results[lang]["correct_indices"], per_lang_results[other]["correct_indices"]) for other in others]
        final_stats[lang]["clc"] = sum(overlaps) / len(overlaps) if overlaps else 0

    return final_stats, specimens

In [ ]:
# --- CONFIGURATION ---
DATA_ROOT = "/content/drive/MyDrive/SNLP_Project/KLAR-DATA-LIBRARY/klar"
# Usable intersection of KLAR and your requested languages
TARGET_LANGS = ["es", "fr", "ru", "zh", "ja", "ar"]

In [ ]:
# Load Model

model_path = "/content/drive/MyDrive/SNLP_Project/models/olmo2_base_4bit"
# model_path =
model_label = "OLMo2-Base"
# model_label =

print("Loading model...")
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(model_path, device_map="auto", trust_remote_code=True)
model.eval()

In [ ]:
# clear mode cache if we want to
# del model; del tokenizer; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# --- 1. RUN EVALS ---
n_shot = 3 # 3 shot eval used un paper
debug_mode = False # True only for making sure everything is working
# Replace with your actual Drive paths
base_stats, base_specimens = run_model_evaluation(model, tokenizer, model_label, n_shot, debug_mode)
# ft_stats   = run_model_evaluation("/content/drive/MyDrive/SNLP_Project/models/olmo2_ft_4bit", "OLMo2-FineTuned")

## Save Results

In [ ]:
import json
from datetime import datetime

# 1. Define the base path
timestamp = datetime.now().strftime("%Y%m%d_%H%M")
# We use a 'final' suffix so you know this is the big run
save_path_final = f"/content/drive/MyDrive/SNLP_Project/results_base_model_FINAL.json"

# 2. Construct the payload
payload = {
    "stats": base_stats,
    "specimens": base_specimens,
    "metadata": {
        "model": "OLMo2-Base",
        "timestamp": timestamp,
        "n_shot": n_shot,
        "debug_mode": debug_mode
    }
}

# 3. Save with 'ensure_ascii=False' to keep non-English characters readable
with open(save_path_final, 'w', encoding='utf-8') as f:
    json.dump(payload, f, indent=4, ensure_ascii=False)

print(f"✅ BIG RUN COMPLETE.")
print(f"📊 Stats and {len(base_specimens)} specimens saved to: {save_path_final}")

## Load Saved Results

In [ ]:
import json

load_path = "/content/drive/MyDrive/SNLP_Project/results_base_model.json"

if os.path.exists(load_path):
    with open(load_path, 'r', encoding='utf-8') as f:
        payload = json.load(f)

    # Extract back into original variable names
    base_stats = payload.get("stats", {})
    base_specimens = payload.get("specimens", [])

    print(f"📂 Data reloaded. Found {len(base_stats)} languages and {len(base_specimens)} specimens.")
else:
    print("⚠️ File not found! Check your Drive path.")

In [ ]:
# check loading is good
if base_specimens:
    test_s = base_specimens[0]
    print(f"Sample Check: {test_s['lang']} | {test_s['subject']} -> {test_s['target_nat']}")

    # Use your previously defined browser to find a "Diverse" sample
    valid_options = find_valid_specimens(base_specimens, 'success')
    print(f"Confirmed: {len(valid_options)} diverse specimens available for plotting.")

## Plotting

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Use the dictionary returned by your evaluation function (e.g., base_stats)
def plot_final_results(stats_dict):
    if not stats_dict:
        print("⚠️ No data found in the stats dictionary!")
        return

    # 1. Setup Figure
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 12))
    langs = sorted(stats_dict.keys())

    # --- FIGURE 8: BARS (ACC vs CLC) ---
    acc_vals = [stats_dict[l]['acc'] for l in langs]
    clc_vals = [stats_dict[l]['clc'] for l in langs]

    x = np.arange(len(langs))
    width = 0.35

    ax1.bar(x - width/2, acc_vals, width, label='Accuracy', color='#99af93', edgecolor='black', alpha=0.8)
    ax1.bar(x + width/2, clc_vals, width, label='CLC', color='#93a2c9', edgecolor='black', alpha=0.8)

    ax1.set_title('Figure 8: Performance & Consistency', fontsize=14, fontweight='bold')
    ax1.set_xticks(x)
    ax1.set_xticklabels([l.upper() for l in langs])
    ax1.set_ylim(0, 1.1)
    ax1.legend()
    ax1.grid(axis='y', linestyle='--', alpha=0.4)

    # --- FIGURE 5: INTERNAL CONSISTENCY CURVES ---
    colors = {'es': 'blue', 'fr': 'orange', 'ru': 'green', 'zh': 'red', 'ja': 'purple', 'ar': 'brown'}

    for lang in langs:
        # Crucial: Access 'avg_curve' which contains the similarities per layer
        curve = stats_dict[lang].get('avg_curve', [])

        if len(curve) > 0:
            ax2.plot(range(len(curve)), curve,
                     label=f'English vs {lang.upper()}',
                     color=colors.get(lang, 'black'),
                     linewidth=2.5, marker='o', markersize=3)
            print(f"✅ Plotted {lang.upper()} curve. Max Similarity: {max(curve):.3f}")
        else:
            print(f"⚠️ No curve data found for {lang.upper()}")

    ax2.set_title('Figure 5: Concept Convergence (Layer-Wise)', fontsize=14, fontweight='bold')
    ax2.set_xlabel('Transformer Layer Number')
    ax2.set_ylabel('Cosine Similarity')

    # Highlight the "Knowledge Zone"
    ax2.axvspan(12, 24, color='gold', alpha=0.1, label='Universal Knowledge Zone')

    ax2.grid(True, linestyle='--', alpha=0.5)
    ax2.legend(loc='lower right')

    plt.tight_layout()
    plt.show()

# Run this with your actual variable
plot_final_results(base_stats)

Observations on Figure 5 (The Curves)
Looking at your curves, there is a very interesting "Dip and Rise":

Layer 10 Dip: All languages hit a similarity floor around Layer 10. This is common in OLMo models where the first third of the model is purely dedicated to linguistic "cleaning" (syntax/grammar).

The Knowledge Zone (Layer 15–25): Notice how French (orange) and Spanish (blue) start climbing significantly higher than Arabic or Russian. This is the English Overlayer in action! The model is finding the universal fact-concept in those layers.

Final Layer Divergence: The sharp rise at the very end (Layer 32) is the model forcing the internal concept back into language-specific tokens for the output head.

# ...

In [ ]:
def get_logit_lens_ranks(prompt, nat, err_or_decoy, eng):
    # Retrieve the specific IDs being used
    id_nat = tokenizer.encode(" " + str(nat).strip(), add_special_tokens=False)[-1]
    id_err = tokenizer.encode(" " + str(err_or_decoy).strip(), add_special_tokens=False)[-1]
    id_eng = tokenizer.encode(" " + str(eng).strip(), add_special_tokens=False)[-1]

    ids = [id_nat, id_err, id_eng]
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    ranks = [[], [], []]

    with torch.no_grad():
        outputs = model(inputs.input_ids, output_hidden_states=True)
        for hs in outputs.hidden_states:
            logits = model.lm_head(model.model.norm(hs[0, -1, :]))
            sorted_idx = torch.argsort(logits, descending=True)
            for i, tid in enumerate(ids):
                ranks[i].append((sorted_idx == tid).nonzero(as_tuple=True)[0].item())

    return ranks, ids

In [ ]:
def plot_logit_lens_v2(specimen, decoy="London"):
    # 1. Determine words
    is_err = specimen['type'] == 'failure'
    wrong_word = specimen.get('model_pred', decoy) if is_err else decoy

    # 2. Get ranks and the actual IDs used
    ranks, ids = get_logit_lens_ranks(
        specimen["prompt"],
        specimen["target_nat"],
        wrong_word,
        specimen["target_en"]
    )

    # 3. COLLISION CHECKER
    id_nat, id_wrong, id_eng = ids
    if id_nat == id_eng:
        print(f"⚠️  WARNING: Native and English share ID {id_nat} ('{specimen['target_nat']}').")
        print("Lines will overlap perfectly. Recommend choosing a different specimen.")

    # 4. PLOTTING
    plt.figure(figsize=(10, 6))
    layers = range(len(ranks[0]))

    plt.plot(layers, np.log1p(ranks[0]), label=f"Native ({specimen['target_nat']})", color='blue', linewidth=3)
    plt.plot(layers, np.log1p(ranks[2]), label=f"English ({specimen['target_en']})", color='gray', linestyle='--')
    plt.plot(layers, np.log1p(ranks[1]), label=f"Error/Decoy ({wrong_word})", color='red', alpha=0.4, linestyle=':')

    plt.gca().invert_yaxis()
    plt.axvspan(12, 24, color='gold', alpha=0.08, label='Knowledge Zone')
    plt.title(f"Figure 5{'b' if is_err else 'a'}: {specimen['subject']} ({specimen['lang'].upper()})")
    plt.ylabel("Log(Rank) - Top is More Probable")
    plt.legend()
    plt.grid(True, alpha=0.1)
    plt.show()

In [ ]:
# Usage:
success_candidates = find_valid_specimens(base_specimens, 'success')
for idx, s in success_candidates:
    print(f"Index {idx}: {s['lang'].upper()} | {s['subject']} -> {s['target_nat']} vs {s['target_en']}")

# Now just pick an index from the list above and plot it:
# plot_logit_lens_v2(base_specimens[SOME_INDEX])

In [ ]:
plot_logit_lens_v2(base_specimens[0])

Cross-Lingual Consistency (CLC):
- Does the model give the same answer in the target language that it gave in English?
- Crucial Detail: In the Wang et al. paper, CLC is specifically measured on the subset of questions where the model was already correct in English.
- Formula: (Correct in Native AND Correct in English) / (Correct in English)